# SatQuery: train the land-cover model (any change, anywhere) on a free Kaggle GPU

One U-Net with nine land-cover classes, trained on Dynamic World expert labels, in a multispectral and a true-colour version. Floods (Sen1Floods11, Bolivia and India included) are only a test. A radar water model is trained too.

**Before you start:** Settings → Accelerator **GPU T4** (not P100), Internet **On**. Then **Save Version → Save & Run All**.

1. Step 1 checks the GPU, the internet and the code (1-2 minutes).
2. Step 2 is a **self-test**: every later step on a few real images (about 5-10 minutes). If something is broken you find out here, not after hours.
3. Steps 3-7 are the real run (about 2-3 hours). Results are saved after every step.

Output tab: `satquery_landcover_models.zip` (models, scores and `run_log.txt`). **If a step fails, send `run_log.txt`**: it holds the real error.

In [ ]:
# Settings and the helper every step uses (nothing to change here)
import collections, json, os, subprocess, sys, time, urllib.parse, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

WORK = Path(os.environ.get("SQ_WORK", "/kaggle/working"))      # kept as the notebook's output
TMP = Path(os.environ.get("SQ_TMP", "/tmp/satquery"))          # downloads, thrown away afterwards
REPO = Path(os.environ.get("SQ_REPO", str(TMP / "SatQuery")))
TRAIN_TILES, TEST_TILES, EPOCHS, RADAR_EPOCHS = [int(x) for x in os.environ.get("SQ_SIZE", "1500,0,40,60").split(",")]   # 0 = all
WORK.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
LOG = WORK / "run_log.txt"
LOG.write_text("")
STATUS = {}

def say(text):
    print(text, flush=True)
    with LOG.open("a", encoding="utf-8") as fh:
        fh.write(text + "\n")

def pack():
    "Zip whatever exists so far, so a later failure never loses finished work."
    with zipfile.ZipFile(WORK / "satquery_landcover_models.zip", "w") as z:
        for f in ("landcover_seg.pt", "landcover_seg.pt.metrics.json", "flood_seg.pt", "flood_seg.pt.metrics.json",
                  "india_test/results.json", "india_test/real_pairs_india.jpg", "run_log.txt"):
            if (WORK / f).exists():
                z.write(WORK / f, f)

def run(name, cmd, required=True, cwd=None):
    "Run a command, show its output live, keep it in run_log.txt, and show the real error if it fails."
    say(f"\n===== {name} =====")
    t0, tail = time.time(), collections.deque(maxlen=40)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        tail.append(line.rstrip())
        say(line.rstrip())
    ok = p.wait() == 0
    STATUS[name] = "ok" if ok else "FAILED"
    say(f"----- {name}: {STATUS[name]} in {(time.time() - t0) / 60:.1f} min")
    if not ok:
        last = tail[-1] if tail else "no output"
        say(f"\n##### {name} FAILED (exit code {p.returncode}). The real error is above; last line: {last}")
        if required:
            pack()
            raise RuntimeError(f"{name} failed: {last}  (full text in run_log.txt)")
    return ok

PY = sys.executable

In [ ]:
# Step 1a. Code: main branch
if "SQ_REPO" not in os.environ:
    if not REPO.exists():
        run("get code", ["git", "clone", "-q", "--depth", "1", "-b", "main", "https://github.com/Vansh-Vaid/SatQuery.git", REPO])
    run("libraries", [PY, "-m", "pip", "install", "-q", "rasterio", "pystac-client", "planetary-computer", "openpyxl"])


In [ ]:
# Step 1b. Check the GPU, the libraries and the internet before anything long starts
CHECK = """
import sys
import numpy, rasterio, pystac_client, planetary_computer, openpyxl, cv2, scipy, matplotlib, torch
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| torch", torch.__version__, "| rasterio", rasterio.__version__)
if not torch.cuda.is_available():
    sys.exit("NO GPU: Settings -> Accelerator -> GPU T4, then Save & Run All again" if "--need-gpu" in sys.argv else 0)
try:
    torch.nn.Conv2d(3, 8, 3).cuda()(torch.rand(1, 3, 64, 64).cuda()).sum().backward()
except Exception as e:
    sys.exit(f"THIS GPU DOES NOT WORK WITH KAGGLE'S PYTORCH ({e}). Choose GPU T4 instead of P100.")
print("GPU ok:", torch.cuda.get_device_name(0))
"""
(TMP / "check.py").write_text(CHECK)
run("GPU and libraries", [PY, TMP / "check.py"] + ([] if "SQ_REPO" in os.environ else ["--need-gpu"]))
for site in ("https://planetarycomputer.microsoft.com/api/stac/v1",
             "https://storage.googleapis.com/storage/v1/b/sen1floods11/o?maxResults=1"):
    for attempt in range(4):
        try:
            urllib.request.urlopen(site, timeout=60).read(100)
            say(f"reachable: {site}")
            break
        except Exception as e:
            say(f"cannot reach {site}: {type(e).__name__}: {e}")
            if attempt == 3:
                pack()
                raise RuntimeError(f"{site} is not reachable. Is Settings -> Internet On? If it is, the site is down: try later.")
            time.sleep(20)

In [ ]:
# Download helper for the Sen1Floods11 hand-labelled flood scenes (public Google Cloud bucket)
BUCKET, HAND = "sen1floods11", "v1.1/data/flood_events/HandLabeled/"

def http(url, timeout=120):
    for attempt in range(5):
        try:
            return urllib.request.urlopen(url, timeout=timeout).read()
        except Exception as e:
            err = e
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"could not download {url}: {type(err).__name__}: {err}") from err

def flood_data(root, per_split=0):
    "The split lists and their chips under root; per_split > 0 takes only that many chips per split (self-test)."
    root, jobs = Path(root), []
    for split in ("train", "valid", "test", "bolivia"):
        name = f"v1.1/splits/flood_handlabeled/flood_{split}_data.csv"
        text = http(f"https://storage.googleapis.com/{BUCKET}/{name}").decode()
        (root / name).parent.mkdir(parents=True, exist_ok=True)
        lines = [x for x in text.splitlines() if "_S1Hand" in x]
        lines = lines[-per_split:] if per_split else lines
        (root / name).write_text("\n".join(lines) + "\n")
        for line in lines:
            chip = line.split(",")[0].strip().split("_S1Hand")[0]
            jobs += [f"{HAND}{k}/{chip}_{k}.tif" for k in ("S1Hand", "S2Hand", "LabelHand")]

    def get(name):
        dst = root / name
        if not (dst.exists() and dst.stat().st_size):
            dst.parent.mkdir(parents=True, exist_ok=True)
            part = dst.with_name(dst.name + ".part")
            part.write_bytes(http(f"https://storage.googleapis.com/{BUCKET}/" + urllib.parse.quote(name)))
            part.replace(dst)

    t0 = time.time()
    with ThreadPoolExecutor(16) as ex:
        list(ex.map(get, jobs))
    say(f"flood scenes: {len(jobs)} files in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Step 2. SELF-TEST: every step on a few real images. All of it must pass before the long run starts.
ST = TMP / "selftest"
say("\n########## SELF-TEST (about 5-10 minutes) ##########")
run("self-test: Dynamic World images", [PY, "training/fetch_dynamic_world.py", "--out", ST / "dw", "--train-tiles", 14,
                                        "--test-tiles", 6, "--workers", 8], cwd=REPO)
flood_data(ST / "floods", per_split=4)
run("self-test: land-cover model", [PY, "training/train_landcover.py", "--data", ST / "dw", "--out", ST / "landcover_seg.pt",
                                    "--sen1floods11", ST / "floods", "--epochs", 2, "--bs", 4], cwd=REPO)
run("self-test: radar model", [PY, "training/train_flood.py", "--root", ST / "floods", "--out", ST / "flood_seg.pt",
                               "--variants", "s1", "--epochs", 2], cwd=REPO)
run("self-test: India", [PY, "eval/real_pair_test.py", "--region", "india", "--sites", "delhi", "--no-agent",
                         "--landcover-ckpt", ST / "landcover_seg.pt", "--data", ST / "pairs", "--out", ST / "india",
                         "--figure", ST / "india" / "fig.jpg"], cwd=REPO)
say("\n########## SELF-TEST PASSED: every step works on real data (its scores mean nothing) ##########")
pack()

In [ ]:
# Step 3. Real data: 1,500 Dynamic World tiles + all test tiles with their Sentinel-2 scenes (40-60 min)
dw = TMP / "dw"
want = {"train": TRAIN_TILES, "test": TEST_TILES or 399}
built = lambda: {s: len(list((dw / s).glob("*.npz"))) for s in want}
for attempt in range(3):          # tiles already built are kept; failed ones are tried again
    run(f"Dynamic World images (pass {attempt + 1})", [PY, "training/fetch_dynamic_world.py", "--out", dw,
        "--train-tiles", TRAIN_TILES, "--test-tiles", TEST_TILES, "--workers", 12], cwd=REPO, required=attempt == 2)
    say(f"tiles built: {built()}")
    if all(built()[s] >= 0.8 * want[s] for s in want):
        break
if any(built()[s] < 0.4 * want[s] for s in want):
    pack()
    raise RuntimeError("too few images were downloaded for a fair model; the reasons are above. Try again later.")

In [ ]:
# Step 4. Real data: Sen1Floods11 (about 5-10 min). Without it the land-cover model still trains.
try:
    flood_data(TMP / "floods", per_split=int(os.environ.get("SQ_FLOOD_LIMIT", 0)))
    floods = True
except Exception as e:
    floods = False
    say(f"flood scenes unavailable ({e}); continuing without the flood test and the radar model")

In [ ]:
# Step 5. Land-cover model, both versions (saved after each one)
run("land-cover model", [PY, "training/train_landcover.py", "--data", dw, "--out", WORK / "landcover_seg.pt", "--epochs", EPOCHS]
    + (["--sen1floods11", TMP / "floods"] if floods else []), cwd=REPO)
pack()

In [ ]:
# Step 6. Radar water model (floods under cloud). A failure here does not lose the land-cover model.
if floods:
    run("radar model", [PY, "training/train_flood.py", "--root", TMP / "floods", "--out", WORK / "flood_seg.pt",
                        "--variants", "s1", "--epochs", RADAR_EPOCHS], cwd=REPO, required=False)
    pack()

In [ ]:
# Step 7. Held-out India test: nine Indian sites scored against ESA WorldCover. Optional, like step 6.
run("India test", [PY, "eval/real_pair_test.py", "--region", "india", "--no-agent", "--landcover-ckpt", WORK / "landcover_seg.pt",
                   "--data", TMP / "pairs", "--out", WORK / "india_test",
                   "--figure", WORK / "india_test" / "real_pairs_india.jpg"], cwd=REPO, required=False)
pack()

In [ ]:
# Step 8. Verdicts
def read(f):
    return json.loads((WORK / f).read_text()) if (WORK / f).exists() else None

lc, fl, ind = read("landcover_seg.pt.metrics.json"), read("flood_seg.pt.metrics.json"), read("india_test/results.json")
say("\n########## RESULTS ##########")
for v, r in (lc or {"variants": {}})["variants"].items():
    say(f"{v:4s} 9-class mIoU {r['test_miou']:.3f} | 4-class accuracy {r['test_acc_4class']:.3f} vs rules "
        f"{r['baseline_acc_4class']:.3f} | flood water IoU {r.get('sen1floods11_test_water_iou')} vs the app's {r.get('sen1floods11_test_baseline_water_iou')} "
        f"(Bolivia {r.get('sen1floods11_bolivia_water_iou')} vs {r.get('sen1floods11_bolivia_baseline_water_iou')}) -> {'BETTER' if r['beats_baseline'] else 'not better'}")
if fl:
    r = fl["variants"]["s1"]
    say(f"radar water IoU {r['test_iou']:.3f} vs the app's {r['baseline_iou']:.3f} (Bolivia {r.get('bolivia_iou')} vs {r.get('bolivia_baseline_iou')}) -> "
        f"{'BETTER' if r['beats_baseline'] else 'not better'}")
if ind:
    say("India, mean accuracy %: " + str({k: v["mean_accuracy_pct"] for k, v in ind["summary"].items()}))
for name, state in STATUS.items():
    say(f"  {state:7s} {name}")
pack()
say("Done. Download satquery_landcover_models.zip from the Output tab.")

## Afterwards
`change_mapper` uses a version only if it is marked BETTER. Before putting one in `checkpoints/`, also check for false change on image pairs taken a few days apart: the first full run failed there (`docs/TECHNICAL.md`).
